In [13]:
import psycopg2,os
import datetime
from psycopg2 import OperationalError,Error
from dotenv import load_dotenv
from rich.console import Console
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
load_dotenv()

console = Console()
class PostgresManager:
    def __init__(self):
        self.conn = None
        try:
            self.conn = psycopg2.connect(
                dbname = os.getenv("dbname"),   
                user = os.getenv("user"),
                password=os.getenv("password"),
                host = os.getenv("host"),
                port = os.getenv("port")
            )
        except OperationalError as e:
            # Bắt các lỗi do sai mật khẩu, sai DB, mất mạng, DB sập...
            console.print(f"❌ [bold red]Lỗi kết nối Database (OperationalError): {e}")
            
        except Error as e:
            #Bắt các lỗi sâu hơn của psycopg2 nếu có
            console.print(f"❌ [bold red]Lỗi hệ thống Psycopg2: {e}")
            
        except Exception as e:
            #Bắt lỗi của Python
            console.print(f"❌ [bold red]Lỗi môi trường Python: {e}")

    def create_tables(self, ticker:str):
        conn = self.conn
        if conn is None:
            console.print(f"[bold red]Không có kết nối DB. Bỏ qua tạo bảng cho {ticker}")
            return

        table_name = ticker.lower()
        try:
            with conn:
                with conn.cursor()as cur:
                   cur.execute(f"""
                    CREATE TABLE IF NOT EXISTS {table_name}(
                        id SERIAL PRIMARY KEY,
                        time TIMESTAMP,
                        open NUMERIC,
                        high NUMERIC,
                        low NUMERIC,
                        close NUMERIC,
                        volume NUMERIC
                    );
                    """)
                    
        except Exception as e:
            console.print(f"❌ [bold red]Lỗi khi kiểm tra/tạo bảng: {e}")

    def save_df(self, df: pd.DataFrame, ticker: str):
            conn = self.conn
            if conn is None: return
            
            # Đảm bảo các cột theo đúng thứ tự bạn muốn insert
            # Lưu ý: vnstock thường trả về các cột ['time', 'open', 'high', 'low', 'close', 'volume']
            # Hãy chắc chắn thứ tự trong df khớp với SQL
            cols = ['open', 'high', 'low', 'close', 'volume', 'time']
            
            # Chuyển DataFrame thành list các tuple
            values = [tuple(x) for x in df[cols].to_numpy()]
            
            table_name = ticker.lower()
            
            try:
                with conn:
                    with conn.cursor() as cur:
                        # Câu lệnh INSERT sử dụng %s làm placeholder cho execute_values
                        query = f"""
                            INSERT INTO {table_name} (open, high, low, close, volume, time)
                            VALUES %s
                        """
                        # execute_values insert cả batch dữ liệu cực nhanh
                        execute_values(cur, query, values)
                        print(f"✅ Đã lưu thành công {len(df)} dòng vào bảng {table_name}")
            except Exception as e:
                console.print(f"❌ [bold red]Lỗi khi lưu dữ liệu: {e}")

In [ ]:
import contextlib
from vnstock import Quote, change_api_key

# 1. Thiết lập API Key (dùng redirect_stdout VÀ ép kiểu utf-8 cho thùng rác)
with open(os.devnull, 'w', encoding='utf-8') as f:
    with contextlib.redirect_stdout(f):
        change_api_key('vnstock_c70b54f88a0e50b1aaaa404488762d85')
db = PostgresManager()

now = datetime.datetime.now()
stocks = ['FTS', 'HCM', 'ORS', 'SSI', 'VIX', 'BSI', 'CTS', 'AGR', 'VDS', 'APG', 'TVS']
# 2. Khởi tạo đối tượng Quote với mã cổ phiếu
for symbol in stocks:
    q = Quote(symbol=symbol)
    # 3. Lấy dữ liệu lịch sử (OHLCV)
    df = q.ohlcv(start='2026-07-04', end=f'{now.strftime("%Y-%m-%d")}', resolution='1D', length=20000)
    db.create_tables(symbol)
    db.save_df(df,symbol.lower())
    # Hiển thị kết quả
    print(df)
db.conn.close()

✅ Đã lưu thành công 25 dòng vào bảng fts
                  time   open   high    low  close   volume
0  2026-07-06 07:00:00  28.00  28.10  26.35  26.90  2958900
1  2026-07-07 07:00:00  26.90  28.40  26.75  28.35  2299600
2  2026-07-08 07:00:00  28.40  28.50  27.75  27.75  1593500
3  2026-07-09 07:00:00  27.40  28.25  26.90  27.60  1897900
4  2026-07-10 07:00:00  27.50  27.65  26.80  26.85  1398900
5  2026-07-13 07:00:00  26.45  26.80  25.00  25.75  3200400
6  2026-07-14 07:00:00  25.50  26.05  25.00  25.65  1087500
7  2026-07-15 07:00:00  25.60  26.15  24.60  25.05  1175600
8  2026-07-16 07:00:00  24.55  25.40  23.95  25.05  2130800
9  2026-07-17 07:00:00  25.00  25.35  24.20  24.20   904900
10 2026-07-20 07:00:00  23.95  23.95  22.55  22.55  2468900
11 2026-07-21 07:00:00  22.60  23.25  22.00  22.00  1370600
12 2026-07-22 07:00:00  22.40  23.25  22.05  22.70  1732800
13 2026-07-23 07:00:00  23.15  23.70  21.40  23.00  1376600
14 2026-07-24 07:00:00  22.55  22.90  21.60  21.60   936700

In [14]:
import contextlib
from vnstock import Quote, change_api_key

# 1. Thiết lập API Key (dùng redirect_stdout VÀ ép kiểu utf-8 cho thùng rác)
with open(os.devnull, 'w', encoding='utf-8') as f:
    with contextlib.redirect_stdout(f):
        change_api_key('vnstock_c70b54f88a0e50b1aaaa404488762d85')
db = PostgresManager()
now = datetime.datetime.now()
q = Quote(symbol="VNINDEX")
# 3. Lấy dữ liệu lịch sử (OHLCV)
df = q.ohlcv(start='2026-07-04', end=f'{now.strftime("%Y-%m-%d")}', resolution='1D', length=20000)
db.save_df(df,"VNINDEX".lower())
# Hiển thị kết quả
print(df)

✅ Đã lưu thành công 25 dòng vào bảng vnindex
                  time     open     high      low    close     volume
0  2026-07-06 07:00:00  1873.24  1873.58  1829.05  1843.50  712579803
1  2026-07-07 07:00:00  1840.69  1848.25  1825.79  1848.25  462997990
2  2026-07-08 07:00:00  1847.42  1855.74  1842.19  1853.70  550621594
3  2026-07-09 07:00:00  1843.01  1853.70  1840.70  1840.70  459418630
4  2026-07-10 07:00:00  1840.33  1845.86  1828.34  1828.34  493196915
5  2026-07-13 07:00:00  1829.50  1829.50  1781.45  1800.54  728451859
6  2026-07-14 07:00:00  1793.23  1806.77  1774.22  1806.63  488594894
7  2026-07-15 07:00:00  1805.93  1806.63  1776.83  1782.12  508398737
8  2026-07-16 07:00:00  1772.52  1804.30  1757.42  1804.24  716663694
9  2026-07-17 07:00:00  1801.89  1804.24  1787.45  1787.45  385810817
10 2026-07-20 07:00:00  1782.93  1787.45  1736.43  1743.51  789163854
11 2026-07-21 07:00:00  1733.44  1750.30  1725.34  1730.56  573145977
12 2026-07-22 07:00:00  1729.33  1736.91  166